In [0]:
%sql
create connection if not exists eq_conn
TYPE HTTP
OPTIONS(
  host ='https://earthquake.usgs.gov',
  port=443,
  base_path= '/earthquakes/feed/v1.0',
  bearer_token ='na'
)

In [0]:
import datetime

print(datetime.datetime.now().strftime('%Y-%m-%d'))

In [0]:
import requests
import json
host ='https://earthquake.usgs.gov'
base_path = '/earthquakes/feed/v1.0'
end_point = '/summary/all_day.geojson'
url = f'{host}{base_path}{end_point}'  ## Hardcoded URL
response = requests.get(url)
data = json.loads(response.text)
print(url)
print(data)


In [0]:
print(data['features'][0]['geometry']['coordinates'])

In [0]:
#parametering URL
import json 
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
conn = w.connections.get('eq_conn')
print(conn)
host = conn.options['host']
print(host)
base_path = conn.options['base_path']
end_point = '/summary/all_day.geojson'  # Using hardcodded as this detail is not present in conn json 
url = f'{host}{base_path}{end_point}'  
print(url)
response = requests.get(url)
data = response.json()
print(data)





In [0]:

dbutils.widgets.text('Catalog_nm','catalog_dev','label')
catalog_name = dbutils.widgets.get('Catalog_nm')

dbutils.widgets.text('Schema_nm','db_eq_dev','label')
schema_name = dbutils.widgets.get('Schema_nm')

print('Catalog_name : ' +catalog_name)
print('Schema_name : ' + schema_name)



In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

In [0]:
%sql
-- Creating new volume
use catalog catalog_dev;
use schema  db_eq_dev;

create volume if not exists `eq_volume`;



In [0]:
##Writting data to a JSON file and then reading it to DataFrame
dbutils.fs.put('/Volumes/catalog_dev/db_eq_dev/eq_volume/eq_data.json', json.dumps(data), True)
# DBTITLE 1,Read the file
df = spark.read.json('/Volumes/catalog_dev/db_eq_dev/eq_volume/eq_data.json')
display(df)

In [0]:
print(type(df))

In [0]:
display(df.selectExpr("explode(features) as feature").select("feature.*"))

In [0]:
from pyspark.sql.functions import explode

df_flat = df.selectExpr("explode(features) as feature").select("feature.*")
df_flat = df_flat.select(
    "*",
    "geometry.*",
    "properties.*"
).drop("geometry", "properties")

display(df_flat)